# 13 · A puffing thermal plume — HDiv-HDG & HDG 🔥🌀

A hot patch on the floor of a tall closed channel drives a **buoyant thermal plume**. Above
a moderate temperature difference the plume stops rising straight: it **meanders and puffs**
— a genuinely **time-dependent** flow. We solve the **Boussinesq** equations with two of
NGSolve's most powerful flow tools:

* an **H(div)-conforming HDG** velocity that is **exactly divergence-free**, and
* a scalar **HDG** temperature,

advanced by an **IMEX** scheme that keeps every implicit operator *constant* — so we
factorise **once** with `sparsecholesky` and reuse it every step.

In non-dimensional form (velocity scaled by the thermal diffusion speed, $T\in[0,1]$):
$$ \tfrac{1}{Pr}\bigl(\partial_t\mathbf u + (\mathbf u\!\cdot\!\nabla)\mathbf u\bigr)
   = \Delta\mathbf u - \nabla p + Ra\,T\,\mathbf e_y,\quad \nabla\!\cdot\!\mathbf u=0,\qquad
   \partial_t T + \mathbf u\!\cdot\!\nabla T = \Delta T . $$

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "anywidget"], check=True)

In [ ]:
from ngsolve import *
from netgen.occ import WorkPlane, OCCGeometry
import numpy as np
from ngsolve.webgui import Draw

In [ ]:
Ra, Pr = 1e6, 0.71                                   # Rayleigh & Prandtl numbers
order, maxh, dt = 2, 0.04, 8e-6                       # FE order, mesh size, time step
W, Hh = 1.0, 2.0                                     # a tall, closed channel (width W, height Hh)

The domain and its boundary conditions at a glance — a tall closed channel with a **hot
patch** on the floor centre, a **cold ceiling**, **insulated** side walls and floor, and
**no-slip** velocity everywhere:

![The thermal-plume channel (W=1 × H=2) — hot patch T=1 on the floor centre, cold ceiling T=0, insulated walls and floor, no-slip velocity on all walls](https://raw.githubusercontent.com/schruste/ngsum2026-colab/colab/data/plume-domain.png)

In [ ]:
# bottom split into three named pieces so the centre strip can be the hot patch
wp = WorkPlane().MoveTo(0, 0)                        # the floor, split bot | hot | bot
wp.LineTo(0.4, 0, name="bot"); wp.LineTo(0.6, 0, name="hot"); wp.LineTo(W, 0, name="bot")
wp.LineTo(W, Hh, name="wall"); wp.LineTo(0, Hh, name="top"); wp.LineTo(0, 0, name="wall")
geo = OCCGeometry(wp.Face(), dim=2)
mesh = Mesh(geo.GenerateMesh(maxh=maxh))
print(f"channel mesh: {mesh.ne} elements   (hot patch on the floor centre)")

In [ ]:
n = specialcf.normal(2)
h = specialcf.mesh_size
def tang(w): return w - (w * n) * n
dS = dx(element_boundary=True)
alpha = 4
walls = "hot|bot|top|wall"                           # no-slip everywhere

## 1. The velocity space — H(div)-conforming HDG

The velocity lives in **`HDiv`** (continuous normal component) paired with a **pressure**
in `L2` one order lower, which makes the discrete velocity **pointwise divergence-free**.
H(div) carries no tangential continuity, so a **facet unknown** (`TangentialFacetFESpace`)
ties the tangential velocity together **weakly**, HDG-style. No-slip walls fix both.

In [ ]:
V    = HDiv(mesh, order=order, dirichlet=walls, dgjumps=True)            # normal-continuous
Vhat = TangentialFacetFESpace(mesh, order=order, dirichlet=walls)        # tangential trace
Q    = L2(mesh, order=order - 1)                                         # pressure
X = V * Vhat * Q
(u, uhat, p), (v, vhat, q) = X.TnT()
gfu = GridFunction(X); 
velocity = gfu.components[0]

## 2. The implicit Stokes operator — assembled once

IMEX treats: 
* the *stiff linear* parts **implicitly**
* and the *cheap nonlinear* parts (convection, buoyancy) **explicitly**.

The implicit momentum part is a **generalised Stokes** operator (which never changes in time):
 * mass $\tfrac{1}{Pr\,\Delta t}$
 * $+$ viscous HDG
 * $+$ incompressibility
 * $+$ a tiny $-\varepsilon\,pq$ regularisation

Note: the discretization is only non-conforming w.r.t. viscosity $\leadsto$ treatment as DG for diffusion

Writing $\langle\cdot,\cdot\rangle_{\partial\mathcal T}=\sum_T\int_{\partial T}$ for the element-boundary
product, $(w)_t=w-(w\!\cdot\!n)\,n$ for the tangential part and $k$ for the order, the **discrete
variational problem** reads: find $(u,\hat u,p)\in V_h\times\hat V_h\times Q_h$ such that for all
$(v,\hat v,q)$
$$
\underbrace{\tfrac{1}{Pr\,\Delta t}(u,v)_\Omega}_{\text{mass}}
+\underbrace{(\nabla u,\nabla v)_\Omega
  -\langle\nabla u\,n,(v-\hat v)_t\rangle_{\partial\mathcal T}
  -\langle\nabla v\,n,(u-\hat u)_t\rangle_{\partial\mathcal T}
  +\tfrac{\alpha k^2}{h}\langle(u-\hat u)_t,(v-\hat v)_t\rangle_{\partial\mathcal T}}_{\text{viscous HDG — symmetric interior penalty}}
\underbrace{-(p,\operatorname{div}v)_\Omega-(q,\operatorname{div}u)_\Omega-\varepsilon(p,q)_\Omega}_{\text{incompressibility (+ reg.)}}
= \ell^{\,n}(v),
$$
with the **explicit** right-hand side $\ell^{\,n}$ (old velocity mass term, convection, buoyancy)
reassembled each step. The viscous block is the **symmetric interior-penalty (SIP)** HDG form:
consistency term, its adjoint, and the $\alpha k^2/h$ penalty tying each element's tangential
velocity to its facet trace $\hat u$. The line-by-line code below is exactly this form:

In [ ]:
eps = 1e-9
a = BilinearForm(X, symmetric=True)
a += 1 / (Pr * dt) * InnerProduct(u, v) * dx
a += InnerProduct(Grad(u), Grad(v)) * dx
a += (-InnerProduct(Grad(u) * n, tang(v - vhat)) - InnerProduct(Grad(v) * n, tang(u - uhat))
      + alpha * order * order / h * InnerProduct(tang(u - uhat), tang(v - vhat))) * dS
a += (-div(u) * q - div(v) * p - eps * p * q) * dx
a.Assemble()
ainv = a.mat.Inverse(X.FreeDofs(), inverse="sparsecholesky")            # factor ONCE, reuse
mass_u = BilinearForm(1 / (Pr * dt) * InnerProduct(u, v) * dx, symmetric=True).Assemble()

## 3. The temperature — a scalar HDG

Temperature lives in **`L2`** with a **facet** unknown for the HDG diffusion. 

* the **hot patch** is $T=1$,
* the **ceiling** $T=0$ (Dirichlet on the facet trace);
* side walls and the rest of the floor are **insulated** (natural
Neumann).

One `BoundaryCF` sets both Dirichlet values at once.

The **discrete variational problem** is the *same* SIP-HDG diffusion, now scalar: find
$(T,\hat T)\in W_h\times\hat W_h$ such that for all $(s,\hat s)$
$$
\tfrac{1}{\Delta t}(T,s)_\Omega
+(\nabla T,\nabla s)_\Omega
-\langle\nabla T\!\cdot\!n,\,s-\hat s\rangle_{\partial\mathcal T}
-\langle\nabla s\!\cdot\!n,\,T-\hat T\rangle_{\partial\mathcal T}
+\tfrac{\alpha k^2}{h}\langle T-\hat T,\,s-\hat s\rangle_{\partial\mathcal T}
= \tfrac{1}{\Delta t}(T^n,s)_\Omega-c_T(u^n;T^n,s),
$$
with the explicit upwind transport $c_T$ defined in the next section.

In [ ]:
Wt   = L2(mesh, order=order)
What = FacetFESpace(mesh, order=order, dirichlet="hot|top")
Y_ = Wt * What
(T, That), (s, shat) = Y_.TnT()
gfT = GridFunction(Y_); temp = gfT.components[0]

aT = BilinearForm(Y_, symmetric=True)
aT += 1 / dt * T * s * dx + Grad(T) * Grad(s) * dx
aT += (-Grad(T) * n * (s - shat) - Grad(s) * n * (T - That)
       + alpha * order * order / h * (T - That) * (s - shat)) * dS
aT.Assemble()
aTinv = aT.mat.Inverse(Y_.FreeDofs(), inverse="sparsecholesky")
mass_T = BilinearForm(1 / dt * T * s * dx, symmetric=True).Assemble()
gfT.components[1].Set(mesh.BoundaryCF({"hot": 1, "top": 0}),
                     definedon=mesh.Boundaries("hot|top"))

## 4. The explicit pieces — convection & buoyancy

Convection is **explicit**: 
* upwind-DG for the temperature transport $\mathbf u\!\cdot\!\nabla T$,
* and the velocity self-advection $(\mathbf u\!\cdot\!\nabla)\mathbf u$,
* buoyancy $Ra\,T\,\mathbf e_y$ enters the momentum right-hand side.

With the upwind trace $T^{\uparrow}=T$ where $u\!\cdot\!n>0$ and $T^{\text{nb}}$ otherwise, the three
explicit forms (all evaluated at the old state and folded into $\ell^{\,n}$ above) are
$$
c_T(u;T,s)=-(T,\,u\!\cdot\!\nabla s)_\Omega+\sum_F\big\langle (u\!\cdot\!n)\,T^{\uparrow},\,[\![s]\!]\big\rangle_F,
\qquad
c_u(u;v)=\tfrac1{Pr}\big((\nabla u)\,u,\,v\big)_\Omega,
\qquad
\ell_{\text{buo}}(v)=Ra\,(T,\,v_y)_\Omega,
$$
where $[\![s]\!]$ is the facet jump. $c_T$ is the standard **upwind-DG** convection of the
temperature; $c_u$ the velocity self-advection.

In [ ]:
convT = BilinearForm(Y_, nonassemble=True)                              # temperature transport
uTn = velocity * n
convT += -T * (velocity * Grad(s)) * dx
convT += uTn * IfPos(uTn, T, T.Other()) * (s - s.Other()) * dx(skeleton=True)

convU = BilinearForm(X, nonassemble=True)                              # velocity self-advection
convU += 1 / Pr * InnerProduct(Grad(u) * u, v) * dx

buoyancy = LinearForm(Ra * temp * v[1] * dx)                           # Ra·T·e_y on the rhs
resT = gfT.vec.CreateVector(); resU = gfu.vec.CreateVector()

## 5. Step in time — and watch the plume whip

One couplied IMEX step: 
* advance the temperature 
* then the velocity

In [ ]:
def progress(i, n):                                    # tiny dependency-free progress bar
    import os, sys
    if os.environ.get("WEBGUI_SCENE_DIR"):             # static-site build: stay silent (no \r spam)
        return
    if (i + 1) % max(1, n // 100) == 0 or i + 1 == n:  # survives JupyterLite / Colab / local
        b = int(28 * (i + 1) / n)
        sys.stdout.write(f"\r  stepping… [{'█'*b}{'·'*(28-b)}] {100*(i+1)//n:3d}%")
        sys.stdout.flush()
        if i + 1 == n:
            sys.stdout.write("\n")

In [ ]:
tend = 0.04
nsteps = int(tend / dt + 0.5)
snap = max(1, nsteps // 12)                                            # ~12 frames (keeps each scene small)
probe = mesh(0.5, 1.2)

draw_mesh = Mesh(geo.GenerateMesh(maxh=0.10))                         # ~460 elements, for the webgui only
VL, TL = VectorH1(draw_mesh, order=1), H1(draw_mesh, order=1)         # light coarse fields to animate
vel_series  = GridFunction(VL, multidim=0)                            # coarse multidim time series each
temp_series = GridFunction(TL, multidim=0)
vdraw, tdraw = GridFunction(VL), GridFunction(TL)
vel_full  = GridFunction(V,  multidim=0)                              # full-resolution series on the
temp_full = GridFunction(Wt, multidim=0)                              # ORIGINAL mesh -> commented Draw below
vx_t, vy_t, ts = [], [], []
with TaskManager():
    for step in range(1, nsteps + 1):
        convT.Apply(gfT.vec, resT)                                     # explicit transport
        gfT.vec.data += aTinv * ((mass_T.mat * gfT.vec - resT).Evaluate() - aT.mat * gfT.vec)
        convU.Apply(gfu.vec, resU)                                     # explicit self-advection
        buoyancy.Assemble()                                            # explicit buoyancy Ra·T·e_y
        gfu.vec.data = ainv * (mass_u.mat * gfu.vec + buoyancy.vec - resU)  # implicit Stokes
        if step % snap == 0:                                          # snapshot BOTH fields as a frame
            tdraw.Set(temp);     temp_series.AddMultiDimComponent(tdraw.vec)
            vdraw.Set(velocity); vel_series.AddMultiDimComponent(vdraw.vec)
            temp_full.AddMultiDimComponent(temp.vec)                   # full-res copies (original grid)
            vel_full.AddMultiDimComponent(velocity.vec)
            vv = velocity(probe); ts.append(step * dt); vx_t.append(vv[0]); vy_t.append(vv[1])
        progress(step - 1, nsteps)
print(f"done: {nsteps} steps,  ‖div u‖ = {sqrt(Integrate(div(velocity)**2, mesh)):.1e}  "
      f"(exactly div-free),  {len(ts)} time frames stored")

## 6. The result — a meandering, puffing plume

The plume rises straight from the hot patch, mushrooms at the top, then loses its symmetry
and **whips from side to side** — an unsteady, self-sustained dance. Both fields are stored
as **multidim time series** — each scene plays on its own (or drag the *multidim* slider).
The two scenes share the same frames, so they animate **in step**.

In [ ]:
Draw(temp_series, draw_mesh, "temperature — time series", min=0, max=1, autoscale=False,
     interpolate_multidim=True, animate=True)

In [ ]:
Draw(vel_series, draw_mesh, "velocity — time series",
     interpolate_multidim=True, animate=True, vectors={"grid_size": 20})

*Both animations above use a **coarse** auxiliary mesh to keep each webgui scene small. The same
series was also collected on the **original** grid (`temp_full`, `vel_full`) — uncomment the cell
below to animate it at full resolution locally. It is a much heavier scene, so we deliberately keep
it out of the built site.*

In [ ]:
# Draw(temp_full, mesh, "temperature — full resolution", min=0, max=1, autoscale=False,
#      interpolate_multidim=True, animate=True)
# Draw(vel_full, mesh, "velocity — full resolution",
#      interpolate_multidim=True, animate=True, vectors={"grid_size": 30})

A higher-resolution offline rendering of the same run (temperature and velocity side by side),
produced separately with **PyVista** (`scripts/render_plume_video.py`). We *embed* it below so it
plays everywhere — local Jupyter, Colab and the website — instead of a bare `<video src>` whose
relative path neither JupyterLab nor the static build resolves:

In [ ]:
from IPython.display import Video, Markdown, display
import os
_vid = "data/plume.mp4"
display(Video(_vid, embed=True, width=600, mimetype="video/mp4",
              html_attributes="controls loop muted playsinline") if os.path.exists(_vid)
        else Markdown("*(the offline `data/plume.mp4` is not shipped in this environment — see the website build)*"))

## Supplementary — where does the time go? (pajetrace)

In [ ]:
import glob, os, html, pathlib
from IPython.display import display, HTML

if sys.platform != "emscripten":                     # threads/tracing unavailable in JupyterLite
    SetNumThreads(4)
    before = {t["name"]: t["time"] for t in Timers()}    # snapshot to scope the timers
    with TaskManager(pajetrace=10**8):                   # the sunburst covers exactly these steps
        for _ in range(20):                              # 20 more IMEX steps, profiled
            convT.Apply(gfT.vec, resT)
            gfT.vec.data += aTinv * ((mass_T.mat * gfT.vec - resT).Evaluate() - aT.mat * gfT.vec)
            convU.Apply(gfu.vec, resU)
            buoyancy.Assemble()
            gfu.vec.data = ainv * (mass_u.mat * gfu.vec + buoyancy.vec - resU)
    traces = sorted(glob.glob("ng*.html"), key=os.path.getmtime)   # the viewer NGSolve just wrote
    if traces:
        doc = html.escape(pathlib.Path(traces[-1]).read_text(), quote=True)
        display(HTML(f'<iframe srcdoc="{doc}" sandbox="allow-scripts" width="100%" height="460" '
                     f'style="border:1px solid #ddd;border-radius:8px" title="pajetrace sunburst"></iframe>'))
    delta = sorted(((t["time"] - before.get(t["name"], 0.0), t["name"]) for t in Timers()), reverse=True)
    print("hottest routines in 20 profiled IMEX steps:")
    for dtt, name in delta[:6]:
        if dtt > 0:
            print(f"  {dtt * 1e3:8.2f} ms  {name[:48]}")
else:
    print("(threaded profiling / pajetrace needs a real OS — run locally or on Colab)")

In [ ]:
# Navigation between units — shown only in a live notebook (Colab / JupyterLite /
# local Jupyter), never in the rendered website (which has its own prev/next nav).
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):          # not the static site build
    _prev = ("12-pedestrian-dynamics", "12 · A crowd heads for coffee 🚶☕")
    _next = ("14-melting-chocolate", "14 · Melting the chocolate 🍫☕")
    def _u(_nb):
        if "google.colab" in sys.modules:
            return "https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/" + _nb + ".ipynb"
        return _nb + ".ipynb"                       # JupyterLite & local: relative .ipynb link
    _parts  = ["⬅️ **Previous:** [%s](%s)" % (_prev[1], _u(_prev[0]))] if _prev else []
    _parts += ["➡️ **Next:** [%s](%s)" % (_next[1], _u(_next[0]))] if _next else []
    from IPython.display import display, Markdown
    display(Markdown(" · ".join(_parts)))